### RAG Pipeline- Data Ingestion to Vector DB pipeline


In [81]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [82]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f" Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: 22.pdf
 Loaded 1 pages

Processing: 52 (1).pdf
 Loaded 28 pages

Processing: 52.pdf
 Loaded 28 pages

Total documents loaded: 57


In [83]:
all_pdf_documents

[Document(metadata={'producer': 'Acrobat Distiller 25.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2025-05-30T13:24:37+05:30', 'author': 'Atlas', 'moddate': '2025-05-30T13:24:37+05:30', 'title': 'Microsoft Word - First Year undertaking_2025-28_Amended (1).docx', 'source': '..\\data\\pdf\\22.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': '22.pdf', 'file_type': 'pdf'}, page_content='UNDERTAKING FOR DOMICILE OF THE STUDENT \n \n \nDate:   \n \n \n \na. I (Name of candidate)  herewith undertake that I have read all the Rules of \nAdmission to (Name of the Programme)  for the academic year 2025-26 and after fully \nunderstanding all the rules & policy, I have filled in this application form for the current academic year. \nb. I am a citizen of (City)  in (State)     in (Country) \n \nc. The information given by me in my application fo rm is true to the best of my knowledge and \nbelief. \nd. If at any later stage, it is found that I have furn

In [84]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [85]:
chunks=split_documents(all_pdf_documents)
chunks

Split 57 documents into 60 chunks

Example chunk:
Content: UNDERTAKING FOR DOMICILE OF THE STUDENT 
 
 
Date:   
 
 
 
a. I (Name of candidate)  herewith undertake that I have read all the Rules of 
Admission to (Name of the Programme)  for the academic year ...
Metadata: {'producer': 'Acrobat Distiller 25.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2025-05-30T13:24:37+05:30', 'author': 'Atlas', 'moddate': '2025-05-30T13:24:37+05:30', 'title': 'Microsoft Word - First Year undertaking_2025-28_Amended (1).docx', 'source': '..\\data\\pdf\\22.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': '22.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Acrobat Distiller 25.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2025-05-30T13:24:37+05:30', 'author': 'Atlas', 'moddate': '2025-05-30T13:24:37+05:30', 'title': 'Microsoft Word - First Year undertaking_2025-28_Amended (1).docx', 'source': '..\\data\\pdf\\22.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': '22.pdf', 'file_type': 'pdf'}, page_content='UNDERTAKING FOR DOMICILE OF THE STUDENT \n \n \nDate:   \n \n \n \na. I (Name of candidate)  herewith undertake that I have read all the Rules of \nAdmission to (Name of the Programme)  for the academic year 2025-26 and after fully \nunderstanding all the rules & policy, I have filled in this application form for the current academic year. \nb. I am a citizen of (City)  in (State)     in (Country) \n \nc. The information given by me in my application fo rm is true to the best of my knowledge and \nbelief. \nd. If at any later stage, it is found that I have furn

### Embedding and Vector Store DB


In [86]:
import numpy as np
print("numpy OK")

numpy OK


In [87]:
from sentence_transformers import SentenceTransformer
print("sentence_transformers OK")

sentence_transformers OK


In [88]:
import chromadb
print("chromadb OK")

chromadb OK


In [89]:
from sklearn.metrics.pairwise import cosine_similarity
print("sklearn OK")

sklearn OK


In [90]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [91]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(
    texts,
    show_progress_bar=False
)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully. Embedding dimension: 384


C:\Users\aashi\AppData\Local\Temp\ipykernel_27056\3691151478.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### Vector

In [92]:
import os
print(os.getcwd())

c:\Users\aashi\Downloads\RAG\notebook


In [93]:
vectorstore = VectorStore()

print(vectorstore.collection)
print(vectorstore.collection.count())

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 120
Collection(name=pdf_documents)
120


In [94]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 120


In [95]:
chunks

[Document(metadata={'producer': 'Acrobat Distiller 25.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2025-05-30T13:24:37+05:30', 'author': 'Atlas', 'moddate': '2025-05-30T13:24:37+05:30', 'title': 'Microsoft Word - First Year undertaking_2025-28_Amended (1).docx', 'source': '..\\data\\pdf\\22.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': '22.pdf', 'file_type': 'pdf'}, page_content='UNDERTAKING FOR DOMICILE OF THE STUDENT \n \n \nDate:   \n \n \n \na. I (Name of candidate)  herewith undertake that I have read all the Rules of \nAdmission to (Name of the Programme)  for the academic year 2025-26 and after fully \nunderstanding all the rules & policy, I have filled in this application form for the current academic year. \nb. I am a citizen of (City)  in (State)     in (Country) \n \nc. The information given by me in my application fo rm is true to the best of my knowledge and \nbelief. \nd. If at any later stage, it is found that I have furn

In [96]:
### Convert Text to embeddings
texts=[doc.page_content for doc in chunks]

#generate the embeddings
embeddings= embedding_manager.generate_embeddings(texts)
# store in vector dc
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 60 texts...
Generated embeddings with shape: (60, 384)
Adding 60 documents to vector store...
Successfully added 60 documents to vector store
Total documents in collection: 180


RAG Retriever from Vector Store


In [97]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [98]:
rag_retriever

In [99]:
rag_retriever.retrieve("What is ABC ID?")

Retrieving documents for query: 'What is ABC ID?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_8600f4af_30',
  'content': 'Step by Step Guide \nABC ID Creation \n \n \n \nPage 28 of 28',
  'metadata': {'producer': 'Microsoft® Word for Microsoft 365',
   'moddate': '2023-09-27T18:35:47+05:30',
   'creator': 'Microsoft® Word for Microsoft 365',
   'doc_index': 30,
   'page_label': '28',
   'file_type': 'pdf',
   'source_file': '52 (1).pdf',
   'author': 'Sridhar Rajendran',
   'source': '..\\data\\pdf\\52 (1).pdf',
   'content_length': 56,
   'creationdate': '2023-09-27T18:35:47+05:30',
   'total_pages': 28,
   'page': 27},
  'similarity_score': 0.46333062648773193,
  'distance': 0.5366693735122681,
  'rank': 1},
 {'id': 'doc_ed1ef801_59',
  'content': 'Step by Step Guide \nABC ID Creation \n \n \n \nPage 28 of 28',
  'metadata': {'doc_index': 59,
   'page_label': '28',
   'content_length': 56,
   'author': 'Sridhar Rajendran',
   'source': '..\\data\\pdf\\52.pdf',
   'source_file': '52.pdf',
   'creationdate': '2023-09-27T18:35:47+05:30',
   'creator': 'Microsoft® Wo

Integration Vector DB Context Pipeline with LLM Output


In [100]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key="grokapikey"


In [101]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [102]:
class GroqLLM:
    def __init__(self, model_name: str = "gemma2-9b-it", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
    

In [103]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: gemma2-9b-it
Groq LLM initialized successfully!


In [104]:
rag_retriever.retrieve("ABC ID")

Retrieving documents for query: 'ABC ID'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_8600f4af_30',
  'content': 'Step by Step Guide \nABC ID Creation \n \n \n \nPage 28 of 28',
  'metadata': {'source': '..\\data\\pdf\\52 (1).pdf',
   'creationdate': '2023-09-27T18:35:47+05:30',
   'total_pages': 28,
   'content_length': 56,
   'doc_index': 30,
   'source_file': '52 (1).pdf',
   'creator': 'Microsoft® Word for Microsoft 365',
   'page_label': '28',
   'file_type': 'pdf',
   'moddate': '2023-09-27T18:35:47+05:30',
   'page': 27,
   'author': 'Sridhar Rajendran',
   'producer': 'Microsoft® Word for Microsoft 365'},
  'similarity_score': 0.3718464970588684,
  'distance': 0.6281535029411316,
  'rank': 1},
 {'id': 'doc_ed1ef801_59',
  'content': 'Step by Step Guide \nABC ID Creation \n \n \n \nPage 28 of 28',
  'metadata': {'content_length': 56,
   'file_type': 'pdf',
   'total_pages': 28,
   'moddate': '2023-09-27T18:35:47+05:30',
   'page_label': '28',
   'source_file': '52.pdf',
   'creator': 'Microsoft® Word for Microsoft 365',
   'producer': 'Microsoft® Wor

Integration Vector DB Context Pipleline LLM Output

In [105]:
from pydantic import SecretStr

In [113]:

import os
from dotenv import load_dotenv
load_dotenv()


True

In [107]:
import langchain
print(langchain.__version__)

1.3.2


In [108]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [109]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="llama-3.3-70b-versatile"
)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [110]:

answer=rag_simple("How to Create ABC ID?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'How to Create ABC ID?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Follow the Step by Step Guide on pages 15 of 28.


Enhanced RAG Output

In [111]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Hard Negative Mining Technqiues", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Hard Negative Mining Technqiues'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Answer: No relevant context found.
Sources: []
Confidence: 0.0
Context Preview: 


In [112]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("How to create ABC ID?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'How to create ABC ID?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...
Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
Step by Step Guide 
ABC ID Creation 
 
 
 
Page 15 of 28

Step by Step Guide 
ABC ID Creation 
 
 
 
Page 15 of 28

Step by Step Guide 
ABC ID Creation 
 
 
 
Page 15 of 28

Question: How to create ABC ID?

Answer:

Final Answer: Follow the Step by Step Guide on pages 15 of 28.

Citations:
[1] 52 (1).pdf (page 14)
[2] 52.pdf (page 14)
[3] 52.pdf (page 14)
Summary: To proceed, refer to the provided instructions on page 15 out of a total of 28 pages. The step-by-step guide on page 15 will walk you through the necessary process in a detailed and sequential manner.
History: {'question': 'How to create ABC ID?', 'answer': 'Follow the Step by Step Guide on pages 15 of 28.', 'sources': [{'source': '52 (1)